# Search Quality Evaluation Tutorial

This notebook demonstrates how to evaluate **search quality** - measuring query effectiveness, result coverage, and credit efficiency.

**Key metrics:**
- Query quality score
- Result coverage
- Deduplication efficiency
- Credit efficiency

## Setup

In [55]:
%pip install -q tavily-agent-toolkit langgraph langchain-openai


[notice] A new release of pip is available: 25.3 -> 26.0
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [56]:
import os
import getpass
from dotenv import load_dotenv

load_dotenv()

if not os.environ.get("TAVILY_API_KEY"):
    os.environ["TAVILY_API_KEY"] = getpass.getpass("TAVILY_API_KEY:\n")

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OPENAI_API_KEY:\n")

In [57]:
from tavily import TavilyClient
from tavily_agent_toolkit import ModelConfig, ModelObject
from tavily_agent_toolkit.evals import compute_search_quality_metrics

tavily_client = TavilyClient(api_key=os.environ["TAVILY_API_KEY"])

judge_config = ModelConfig(
    model=ModelObject(
        model="gpt-5-mini",
        api_key=os.environ["OPENAI_API_KEY"],
    ),
)

## Example 1: Basic vs Advanced Search Depth

Same query, two depths. **Basic** returns NLP-summarized content (1 credit). **Advanced** returns reranked chunks with higher relevance (2 credits).

In [58]:
from IPython.display import display

query = "solar panel energy efficiency residential cost 2025"

# --- Basic depth (1 credit) ---
basic_response = tavily_client.search(
    query=query,
    search_depth="basic",
    max_results=5,
    include_usage=True,
)

# --- Advanced depth (2 credits) ---
advanced_response = tavily_client.search(
    query=query,
    search_depth="advanced",
    max_results=5,
    chunks_per_source=3,
    include_usage=True,
)

print("=== Basic Results ===")
display(basic_response)
print("\n=== Advanced Results ===")
display(advanced_response)

=== Basic Results ===


{'query': 'solar panel energy efficiency residential cost 2025',
 'response_time': 1.41,
 'follow_up_questions': None,
 'answer': None,
 'images': [],
 'results': [{'url': 'https://cedarcreekenergy.com/how-has-the-price-and-efficiency-of-solar-panels-changed-over-time-2025-update/',
   'title': 'Solar Panel Price & Efficiency Trends: 2025 Update',
   'content': 'In 2025, the average efficiency of solar panels for home installations ranges from 18% to 22%, with some premium models reaching even higher efficiencies.',
   'score': 0.999985,
   'raw_content': None},
  {'url': 'https://solartechonline.com/blog/solar-panel-cost-1500-square-foot-house/',
   'title': 'Solar Panel Cost For 1,500 Sq Ft House: Complete 2025 Guide',
   'content': 'Discover exact solar panel costs for 1500 sq ft homes in 2025. Get pricing breakdowns, financing options, and real savings examples.',
   'score': 0.99978,
   'raw_content': None},
  {'url': 'https://www.solarreviews.com/blog/how-has-the-price-and-effici


=== Advanced Results ===


{'query': 'solar panel energy efficiency residential cost 2025',
 'response_time': 3.0,
 'follow_up_questions': None,
 'answer': None,
 'images': [],
 'results': [{'url': 'https://solareducators.info/blog/solar-panel-installation-guide-2025',
   'title': 'Solar Panel Installation Guide 2025 | Cost & ROI',
   'content': "In 2025, the momentum behind residential and commercial solar energy is undeniable. According to the Solar Energy Industries Association (SEIA), there's been a remarkable 24.7% increase in nationwide solar energy production over the past year, and more property owners than ever are exploring the long-term benefits of generating their own clean power. The average 12-kilowatt (kW) residential solar system, costing approximately $29,649 before incentives according to EnergySage's 2025 Solar Marketplace Report, is not just an eco-friendly upgrade but a significant financial investment that can yield savings of $37,000 to $148,000 over a 25-year lifespan. This comprehensive 

In [ ]:
# Evaluate both with the search quality metric
research_task = "What is the energy efficiency and cost of solar panels for residential use?"

basic_quality = await compute_search_quality_metrics(
    research_task=research_task,
    queries=[query],
    results=basic_response["results"],
    judge_model_config=judge_config,
    credits_used=basic_response.get("usage", {}).get("credits", 0),
)

advanced_quality = await compute_search_quality_metrics(
    research_task=research_task,
    queries=[query],
    results=advanced_response["results"],
    judge_model_config=judge_config,
    credits_used=advanced_response.get("usage", {}).get("credits", 0),
)

# Average relevance scores from Tavily
basic_avg_score = sum(r["score"] for r in basic_response["results"]) / len(basic_response["results"])
advanced_avg_score = sum(r["score"] for r in advanced_response["results"]) / len(advanced_response["results"])

print("Basic vs Advanced — Same Query")
print("=" * 60)
print(f"{'Metric':<25} {'Basic':<18} {'Advanced':<18}")
print(f"{'-'*25} {'-'*18} {'-'*18}")
print(f"{'Avg Relevance Score':<25} {basic_avg_score:<18.3f} {advanced_avg_score:<18.3f}")
print(f"{'Coverage':<25} {basic_quality.result_coverage:<18.1%} {advanced_quality.result_coverage:<18.1%}")
print(f"{'Dedup Efficiency':<25} {basic_quality.deduplication_efficiency:<18.1%} {advanced_quality.deduplication_efficiency:<18.1%}")
print(f"{'Results':<25} {basic_quality.results_count:<18} {advanced_quality.results_count:<18}")
print(f"{'Credits Used':<25} {basic_response.get('usage',{}).get('credits',0):<18} {advanced_response.get('usage',{}).get('credits',0):<18}")

Basic vs Advanced — Same Query
Metric                    Basic              Advanced          
------------------------- ------------------ ------------------
Avg Relevance Score       1.000              0.875             
Coverage                  62.0%              60.0%             
Dedup Efficiency          100.0%             100.0%            
Credit Efficiency         0.4500             0.2825            
Results                   5                  5                 
Credits Used              1                  2                 


In [54]:
# Compare content type: basic returns NLP summaries, advanced returns reranked chunks
print("Content comparison (first result):\n")

print("--- Basic (NLP summary) ---")
print(basic_response["results"][0]["content"][:300])
print()

print("--- Advanced (reranked chunks) ---")
print(advanced_response["results"][0]["content"][:300])

Content comparison (first result):

--- Basic (NLP summary) ---
In 2025, the average efficiency of solar panels for home installations ranges from 18% to 22%, with some premium models reaching even higher efficiencies.

--- Advanced (reranked chunks) ---
In 2025, the momentum behind residential and commercial solar energy is undeniable. According to the Solar Energy Industries Association (SEIA), there's been a remarkable 24.7% increase in nationwide solar energy production over the past year, and more property owners than ever are exploring the lon


## Example 2: Optimizing Query Strategy
Configure the query to provide resutls that give best coverage to your original task/goal

In [71]:
# Compare different query strategies
task = "Latest developments in mRNA vaccine technology"

# Strategy 1: Single broad query
single_query = ["mRNA vaccine developments"]
single_results = tavily_client.search(query=single_query[0], max_results=10, include_usage=True)

# Strategy 2: Multiple specific queries
multi_queries = [
    "mRNA vaccine clinical trials",
    "mRNA technology new applications",
    "mRNA manufacturing advances",
]
multi_results = []
for q in multi_queries:
    r = tavily_client.search(query=q, max_results=3)
    multi_results.extend(r["results"])

In [72]:
# Evaluate both strategies
single_quality = await compute_search_quality_metrics(
    research_task=task,
    queries=single_query,
    results=single_results["results"],
    judge_model_config=judge_config,
)

multi_quality = await compute_search_quality_metrics(
    research_task=task,
    queries=multi_queries,
    results=multi_results,
    judge_model_config=judge_config,
)

print("Strategy Comparison")
print("="*60)
print(f"{'Metric':<25} {'Single Query':<18} {'Multi-Query':<18}")
print(f"{'-'*25} {'-'*18} {'-'*18}")
print(f"{'Query Quality':<25} {single_quality.query_quality_score:<18.2f} {multi_quality.query_quality_score:<18.2f}")
print(f"{'Coverage':<25} {single_quality.result_coverage:<18.1%} {multi_quality.result_coverage:<18.1%}")
print(f"{'Results':<25} {single_quality.results_count:<18} {multi_quality.results_count:<18}")
print(f"{'Unique Results':<25} {single_quality.unique_results_count:<18} {multi_quality.unique_results_count:<18}")

Strategy Comparison
Metric                    Single Query       Multi-Query       
------------------------- ------------------ ------------------
Query Quality             0.18               0.64              
Coverage                  35.0%              60.0%             
Results                   10                 9                 
Unique Results            10                 9                 


In [73]:
# Inspect per-query feedback and improvement suggestions

def print_query_feedback(label, quality_result):
    print(f"\n{label}")
    print("-" * len(label))
    for detail in quality_result.query_details:
        print(f"Query: {detail['query']}")
        print(f"Overall score: {detail['overall_score']:.2f}")
        if detail.get("reasoning"):
            print(f"Reasoning: {detail['reasoning']}")
        suggestions = detail.get("suggestions", [])
        if suggestions:
            print("Suggestions:")
            for s in suggestions:
                print(f"  - {s}")
        else:
            print("Suggestions: (none)")
        print()

print_query_feedback("Multi-Query Feedback", multi_quality)



Multi-Query Feedback
--------------------
Query: mRNA vaccine clinical trials
Overall score: 0.65
Suggestions:
  - Add a timeframe or qualifier to target the latest results, e.g. "recent" or years: "mRNA vaccine clinical trials 2023-2025"
  - Specify what you want from trials: "results", "ongoing", "phase 3", or disease area (e.g. "cancer", "influenza") to reduce noise
  - Include data sources or registries if relevant: "clinicaltrials.gov mRNA vaccine" or "recent trial results preprint"

Query: mRNA technology new applications
Overall score: 0.45
Suggestions:
  - Be more specific about the type of applications: e.g. "mRNA vaccines for cancer", "mRNA therapeutics for rare diseases", or "mRNA for protein replacement"
  - Include keywords that appear in technical literature: "self-amplifying mRNA (saRNA)", "delivery systems", "lipid nanoparticle (LNP)"
  - Add a recency filter: "new applications 2024" or "recent developments mRNA applications"

Query: mRNA manufacturing advances
Overall